# Lecture 10 Activity – Which vehicles strike the pedestrians who die?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

In lecture, half of the 3,636 people killed in New York's traffic crashes
since July 2012 were pedestrians. Over the same years, sport utility vehicles
(SUVs) and pickup trucks became a larger share of the new vehicles Americans
buy. Their taller front ends strike a pedestrian higher on the body. That
raises the question of whether they are deadlier for people on foot.

These data cannot measure how dangerous each vehicle is: they hold no count of
the vehicles on the road. They can show which vehicles appear in the crashes
that killed pedestrians, and whether that mix changed.

> **In crashes that killed a pedestrian, which kinds of vehicle were involved,
> and did the mix change between 2012–2018 and 2019–2026?**

### Reminders

- This activity needs `python-duckdb`. If `import duckdb` fails, run
  `conda activate qss20` and then `conda install python-duckdb` in a terminal.
- `persons`: one row per person killed, with `person_type` and `age`.
  `vehicles`: one row per vehicle in a fatal crash, with `vehicle_type` as the
  police officer typed it. Both link to a crash by `collision_id`.
- Many crashes involve more than one vehicle. Decide which crashes let you say
  which vehicle was involved before you count.
- Write each query with `con.sql('''...''').df()`. Print counts after every
  join.
- In the starter cells, replace each `...` with code.

In [ ]:
import json
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_rows', 30)

RAW = Path('../../data/raw')
con = duckdb.connect()   # a database in memory, for this notebook only

con.execute(f'''
CREATE TABLE persons AS
SELECT CAST(collision_id AS INTEGER) AS collision_id,
       CAST(crash_date AS DATE)      AS crash_date,
       person_type,
       CAST(person_age AS INTEGER)   AS age
FROM read_json_auto('{RAW}/nyc/persons-killed-page-*.json');

CREATE TABLE vehicles AS
SELECT CAST(collision_id AS INTEGER) AS collision_id, vehicle_type
FROM read_json_auto('{RAW}/nyc/fatal-crash-vehicles-batch-*.json');
''')

# Check: 3,636 persons killed and 5,961 vehicles.
con.sql('SELECT COUNT(*) FROM persons').fetchone()[0], \
    con.sql('SELECT COUNT(*) FROM vehicles').fetchone()[0]

## Task 1 · How many vehicles per crash?

Count the vehicles in each crash, then count the crashes by number of
vehicles: one row per number of vehicles, with the number of crashes.

How many fatal crashes involve exactly one vehicle? In a crash with two
vehicles that killed a pedestrian, can these tables tell you which vehicle
struck the pedestrian?



<details><summary>Hint</summary>

Two steps. First `GROUP BY collision_id` with `COUNT(*) AS n_vehicles`; name
that step with `WITH per_crash AS (...)`. Then group `per_crash` by
`n_vehicles`.

</details>

In [ ]:
con.sql('''
WITH per_crash AS (
    SELECT collision_id, COUNT(*) AS n_vehicles
    FROM vehicles
    GROUP BY ...
)
SELECT n_vehicles, COUNT(*) AS crashes
FROM per_crash
GROUP BY ...
ORDER BY n_vehicles
''').df()

*Your answer:*

## Task 2 · One pedestrian, one vehicle

Build a table `ped_vehicle` with one row per pedestrian killed in a
single-vehicle crash: the pedestrian's `crash_date` and `age`, and the
vehicle's `vehicle_type`.

How many of the 1,805 pedestrians killed remain? Why must the restriction to
one vehicle come **before** the join to `vehicles`?



<details><summary>Hint</summary>

Name the single-vehicle crashes with `HAVING COUNT(*) = 1` in a `WITH` step.
Then join three tables: the pedestrians (`WHERE person_type = 'Pedestrian'`),
the single-vehicle crashes, and `vehicles`, each `USING (collision_id)`. Store
the result with `CREATE TABLE ped_vehicle AS`.

One `JOIN` can follow another; each adds a table. Joining to `single`, a list
of crash ids, keeps only the crashes on that list, so it works as a filter.
`COUNT(DISTINCT collision_id)` counts each crash once.

</details>

In [ ]:
con.execute('''
CREATE OR REPLACE TABLE ped_vehicle AS
WITH single AS (
    SELECT collision_id
    FROM vehicles
    GROUP BY collision_id
    HAVING ...
)
SELECT p.collision_id, p.crash_date, p.age, v.vehicle_type
FROM persons AS p
JOIN single USING (collision_id)
JOIN vehicles AS v USING (...)
WHERE ...
''')
con.sql('''
SELECT COUNT(*) AS rows, COUNT(DISTINCT collision_id) AS crashes
FROM ped_vehicle
''').df()

*Your answer:*

## Task 3 · Group the vehicle types

List every `vehicle_type` in `ped_vehicle` with its count. Then write a
`CASE` expression that puts each into one class: `SUV`, `car`, `pickup
truck`, `truck or van`, `bus`, `taxi`, `motorcycle`, `bike or scooter`, or
`other or unknown`. Store the result as a new table, `ped_class`: the columns
of `ped_vehicle` plus `vclass`. Then print the count of each class. A new
table, instead of replacing `ped_vehicle`, lets you rerun the cell after each
fix.

Which spellings did you have to combine? How many rows fall into `other or
unknown`?



<details><summary>Hint</summary>

Compare in capitals so that `'Sedan'` and `'SEDAN'` match:
`UPPER(vehicle_type) LIKE '%SPORT UTILITY%'`. Order matters in `CASE`: put
`PICK` before the general `TRUCK` test, or pickups become trucks. A `NULL`
type fails every test and falls to `ELSE`.

</details>

In [ ]:
con.sql('''
SELECT vehicle_type, COUNT(*) AS n
FROM ped_vehicle
GROUP BY vehicle_type
ORDER BY n DESC
LIMIT 15
''').df()

In [ ]:
con.execute('''
CREATE OR REPLACE TABLE ped_class AS
SELECT collision_id, crash_date, age, vehicle_type,
       CASE
           WHEN UPPER(vehicle_type) LIKE '%SPORT UTILITY%'
             OR UPPER(vehicle_type) LIKE '%STATION WAGON%' THEN 'SUV'
           WHEN UPPER(vehicle_type) IN ('SEDAN', 'PASSENGER VEHICLE')
                THEN 'car'
           ...
           ELSE 'other or unknown'
       END AS vclass
FROM ped_vehicle
''')
con.sql('SELECT vclass, COUNT(*) AS n FROM ped_class GROUP BY vclass').df()

*Your answer:*

## Task 4 · ❓ Did the mix change?

Count the pedestrians killed by `vclass` and by period: `'2012-2018'` or
`'2019-2026'`, from the year of `crash_date`. Then, in pandas, compute each
class's share of the pedestrians killed in its period, and put the two
periods side by side.

Which classes gained share, and which lost?



<details><summary>Hint</summary>

The period is another `CASE`: `CASE WHEN year(crash_date) <= 2018 THEN
'2012-2018' ELSE '2019-2026' END`. Group by both columns. In pandas,
`mix.groupby('period')['n'].transform('sum')` gives each row its period's
total; divide `n` by it. `.pivot(index='vclass', columns='period',
values='share')` puts the periods side by side.

</details>

In [ ]:
mix = con.sql('''
SELECT vclass,
       CASE WHEN ... THEN '2012-2018'
            ELSE '2019-2026' END AS period,
       COUNT(*) AS n
FROM ped_class
GROUP BY ...
''').df()

mix = mix.assign(share=...)
shares = mix.pivot(index='vclass', columns='period', values='share')
shares.assign(change=shares['2019-2026'] - shares['2012-2018']).round(3)

*Your answer:*

## Task 5 · What the counts cannot tell you

Write three or four sentences. Do these results show that SUVs are more
dangerous to pedestrians than cars? What would you need to know, and what did
your choices in Tasks 2 and 3 leave out?

*Your answer:*